# Regulatory Document Processing Pipeline with Textract

Universal document processing pipeline using AWS Textract and Bedrock for any file format.

## 1. Dependencies & Imports

Import necessary libraries for AWS integration (Boto3), NLP processing (Transformers, PyTorch), document parsing (BeautifulSoup), and data handling.

In [87]:
import boto3
import json
import time
import os
from typing import Dict, Any
from botocore.exceptions import ClientError
from bs4 import BeautifulSoup
from transformers import AutoTokenizer, AutoModel, pipeline
import torch

## 2. RegulatoryDocumentProcessor Class

Main processing class that orchestrates:
- **Text Extraction**: Handles multiple formats (HTML, XML, PDF, images) using AWS Textract and BeautifulSoup
- **Classification**: Uses LegalBERT to classify regulations into categories (Environmental, Financial, Trade, Privacy, Labor, Tax)
- **Key Information Extraction**: Leverages Claude via Bedrock API to extract critical compliance details

In [88]:
class RegulatoryDocumentProcessor:
    def __init__(self):
        self.bedrock = boto3.client('bedrock-runtime', region_name='us-east-1')
        self.textract = boto3.client('textract', region_name='us-east-1')
        
        # Load LegalBERT for legal document classification
        print("📚 Loading LegalBERT model for legal analysis...")
        self.legal_tokenizer = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")
        self.legal_model = AutoModel.from_pretrained("nlpaueb/legal-bert-base-uncased")
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.legal_model.to(self.device)
        print("✅ LegalBERT model loaded successfully")
        
    def extract_text_from_file(self, file_path: str) -> str:
        """Extract text from any local file using Textract or BeautifulSoup"""
        print(f"📄 Extracting text from: {file_path.split('/')[-1]}")
        
        _, ext = os.path.splitext(file_path.lower())
        print(f"   File extension detected: {ext}")
        
        if ext in ['.html', '.xml']:
            # Use BeautifulSoup for HTML/XML FIRST (before Textract)
            print("   Using BeautifulSoup for HTML/XML parsing...")
            try:
                with open(file_path, 'r', encoding='utf-8') as file:
                    soup = BeautifulSoup(file, 'html.parser')
                    text = soup.get_text(separator='\n')
                print(f"✅ Extracted {len(text):,} characters using BeautifulSoup")
                return text
            except Exception as e:
                print(f"❌ Error parsing HTML/XML: {e}")
                return ""
        
        elif ext in ['.pdf', '.jpg', '.jpeg', '.png', '.tiff', '.tif']:
            # Use Textract for images and PDFs
            print("   Using Textract for document extraction...")
            try:
                with open(file_path, 'rb') as file:
                    document_bytes = file.read()
                
                response = self.textract.detect_document_text(
                    Document={'Bytes': document_bytes}
                )
                
                text = ""
                for block in response['Blocks']:
                    if block['BlockType'] == 'LINE':
                        text += block['Text'] + "\n"
                
                print(f"✅ Extracted {len(text):,} characters using Textract")
                return text
            except ClientError as e:
                if e.response['Error']['Code'] == 'UnsupportedDocumentException':
                    print("❌ Unsupported document format. Textract supports JPEG, PNG, PDF, and TIFF files.")
                    return ""
                else:
                    raise
        
        else:
            print(f"❌ Unsupported file format: {ext}. Supported: PDF, images (JPEG/PNG/TIFF), HTML, XML.")
            return ""
    
    def classify_regulation_with_legalbert(self, text: str) -> Dict[str, Any]:
        """Classify regulation using LegalBERT for legal document understanding"""
        print("🔍 Classifying document using LegalBERT...")
        
        # Truncate text to avoid token limit (512 tokens max for BERT)
        truncated_text = text[:2000]
        
        try:
            # Tokenize input
            inputs = self.legal_tokenizer(
                truncated_text,
                return_tensors="pt",
                truncation=True,
                max_length=512,
                padding=True
            ).to(self.device)
            
            # Get embeddings from LegalBERT
            with torch.no_grad():
                outputs = self.legal_model(**inputs)
                embeddings = outputs.last_hidden_state.mean(dim=1)
            
            # Use embedding norm as confidence measure
            confidence = min(torch.norm(embeddings).item() / 10, 1.0)
            print(f"📋 LegalBERT confidence score: {confidence:.2%}")
            
            # Use Claude for contextual classification
            prompt = f"""Based on this regulatory document excerpt, classify it into ONE category:
            - Environmental
            - Financial
            - Trade
            - Privacy
            - Labor
            - Tax
            
            Document excerpt: {truncated_text}
            
            Return only the category name and a brief 1-2 sentence explanation."""
            
            response = self.bedrock.invoke_model(
                modelId='anthropic.claude-sonnet-4-5-20250929-v1:0',
                body=json.dumps({
                    'anthropic_version': 'bedrock-2023-05-31',
                    'max_tokens': 100,
                    'messages': [{'role': 'user', 'content': prompt}]
                })
            )
            
            result = json.loads(response['body'].read())
            classification = result['content'][0]['text'].strip()
            print(f"📋 Document classification: {classification}")
            
            return {
                'category': classification,
                'legalbert_confidence': confidence
            }
        
        except Exception as e:
            print(f"⚠️ Classification error: {e}. Falling back to basic extraction.")
            return {'category': 'Unknown', 'legalbert_confidence': 0.0}
    
    def extract_key_info(self, text: str, category: str) -> Dict[str, Any]:
        """Extract key regulatory information using Claude"""
        print(f"🔎 Extracting key information for {category} regulation...")
        prompt = f"""Extract key information from this {category} regulation:
        
        Required fields:
        - title: Document title
        - effective_date: When it takes effect
        - affected_sectors: Which business sectors are impacted
        - key_requirements: Main compliance requirements (max 3)
        - penalties: Potential penalties for non-compliance
        
        Document: {text[:4000]}
        
        Return as JSON format."""
        
        response = self.bedrock.invoke_model(
            modelId='anthropic.claude-3-haiku-20240307-v1:0',
            body=json.dumps({
                'anthropic_version': 'bedrock-2023-05-31',
                'max_tokens': 500,
                'messages': [{'role': 'user', 'content': prompt}]
            })
        )
        
        result = json.loads(response['body'].read())
        content = result['content'][0]['text']
        print("✅ Key information extracted")
        
        try:
            return json.loads(content)
        except:
            return {'raw_content': content}
    
    def process_document(self, file_path: str) -> Dict[str, Any]:
        """Process any document format"""
        print(f"🚀 Starting document processing pipeline...")
        text = self.extract_text_from_file(file_path)
        
        if not text:
            print("❌ Failed to extract text from document")
            return None
        
        classification = self.classify_regulation_with_legalbert(text)
        key_info = self.extract_key_info(text, classification['category'])
        print("🎉 Processing completed successfully!")
        
        return {
            'document_id': file_path.split('/')[-1],
            'category': classification['category'],
            'legalbert_confidence': classification.get('legalbert_confidence', 0.0),
            'extracted_info': key_info,
            'processing_status': 'completed'
        }

## Process Any Document Format

## 3. Execute Full Processing Pipeline

Run the complete document processing workflow on a sample regulatory file. The pipeline will:
1. Locate the document file
2. Extract text based on file format
3. Classify the regulation type
4. Extract key compliance information
5. Return structured results

In [90]:
# Update the document_path to point to a valid local file and add a simple fallback search.
# The pipeline now supports HTML/XML alongside PDF and image formats.
# Adjust the candidate paths list if your files live elsewhere

candidate_paths = [
    os.path.join('directives', '1.DIRECTIVE (UE) 20192161 DU PARLEMENT EUROPÉEN ET DU CONSEIL.html'),
    # os.path.join('directive', '1.DIRECTIVE (UE) 20192161 DU PARLEMENT EUROPÉEN ET DU CONSEIL.html'),

]

document_path = None
for path in candidate_paths:
    if os.path.exists(path):
        document_path = path
        break

if not document_path:
    raise FileNotFoundError(
        "Unable to locate the sample document. Update candidate_paths with a valid file location."
    )

print(f"Using document: {document_path}")

processor = RegulatoryDocumentProcessor()
try:
    result = processor.process_document(document_path)
except FileNotFoundError:
    print(f"Error: File not found at {document_path}. Please check the path and ensure the file exists.")
    result = None

if result:
    print("\nProcessing Results:")
    print(f"Document: {result['document_id']}")
    print(f"Category: {result['category']}")
    print(f"Status: {result['processing_status']}")
    print("\nExtracted Information:")
    print(json.dumps(result['extracted_info'], indent=2))

Using document: directives/1.DIRECTIVE (UE) 20192161 DU PARLEMENT EUROPÉEN ET DU CONSEIL.html
📚 Loading LegalBERT model for legal analysis...
✅ LegalBERT model loaded successfully
🚀 Starting document processing pipeline...
📄 Extracting text from: 1.DIRECTIVE (UE) 20192161 DU PARLEMENT EUROPÉEN ET DU CONSEIL.html
   File extension detected: .html
   Using BeautifulSoup for HTML/XML parsing...
✅ LegalBERT model loaded successfully
🚀 Starting document processing pipeline...
📄 Extracting text from: 1.DIRECTIVE (UE) 20192161 DU PARLEMENT EUROPÉEN ET DU CONSEIL.html
   File extension detected: .html
   Using BeautifulSoup for HTML/XML parsing...
✅ Extracted 111,996 characters using BeautifulSoup
🔍 Classifying document using LegalBERT...
📋 LegalBERT confidence score: 90.80%
✅ Extracted 111,996 characters using BeautifulSoup
🔍 Classifying document using LegalBERT...
📋 LegalBERT confidence score: 90.80%
⚠️ Classification error: An error occurred (ValidationException) when calling the InvokeMode